# Fine-tuning PaliGemma-3B for Multi-Turn Visual QA
### QLoRA Fine-tuning on Google Colab (T4 GPU)

**Project:** DLCV Course — Multi-Turn Conversational Visual Question Answering  
**Model:** `google/paligemma-3b-pt-224` (3B parameters)  
**Method:** QLoRA (4-bit quantization + LoRA adapters)  
**Dataset:** VisualDialog v1.0 + LLaVA-Instruct-150K (conversation subset)  

---

**Academic framing:**  
We fine-tune a 3B vision-language model on multi-turn visual dialog data using QLoRA,  
enabling conversation-aware visual question answering on consumer hardware.  
We compare zero-shot baseline vs. fine-tuned model across 4 question types and 4 history lengths.


## Step 0 — Verify GPU

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')
# Expected: Tesla T4, 15.8 GB

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install -q transformers==4.44.0 peft==0.12.0 bitsandbytes==0.43.3 \
             trl==0.9.6 accelerate==0.33.0 datasets==2.21.0 \
             evaluate rouge-score pycocoevalcap Pillow requests
print('✅ Dependencies installed')

## Step 2 — Mount Google Drive (to save model)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/dlcv_paligemma'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Save directory: {SAVE_DIR}')

## Step 3 — HuggingFace Login (for PaliGemma access)

In [ ]:
# PaliGemma requires accepting license on HuggingFace
# Go to: https://huggingface.co/google/paligemma-3b-pt-224
# Click 'Agree and access repository'
# Then create a token at: https://huggingface.co/settings/tokens

from huggingface_hub import login
HF_TOKEN = ''  # ← paste your HF token here
login(token=HF_TOKEN)
print('✅ Logged in to HuggingFace')

## Step 4 — Load Dataset

In [ ]:
import json
import requests
import random
from datasets import load_dataset
from PIL import Image
from io import BytesIO

# ── LLaVA-Instruct conversation subset ─────────────────────────────────────
# 58K multi-turn conversations from LLaVA-Instruct-150K
print('Loading LLaVA-Instruct conversation data...')
llava_ds = load_dataset(
    'liuhaotian/LLaVA-Instruct-150K',
    data_files='conversation_58k.json',
    split='train'
)
print(f'LLaVA conversations: {len(llava_ds)} samples')

# ── VisualDialog v1.0 ───────────────────────────────────────────────────────
# Official download — 10-round visual dialogs over COCO images
print('\nLoading VisualDialog v1.0...')
vdial_url = 'https://www.dropbox.com/s/ix8keeudqrd8hn8/visdial_1.0_train.zip?dl=1'
# Alternative: load from HuggingFace
vdial_ds = load_dataset('jxu124/visdial', split='train', trust_remote_code=True)
print(f'VisualDialog: {len(vdial_ds)} dialogs')

In [ ]:
# ── Build unified training set ──────────────────────────────────────────────
# Format: list of {image_url_or_path, conversations: [{role, content}]}

LLAVA_SAMPLES  = 1000   # from LLaVA-Instruct
VDIAL_SAMPLES  = 1500   # from VisualDialog
EVAL_SPLIT     = 0.1    # 10% for validation

random.seed(42)

def format_llava_sample(sample):
    """Convert LLaVA format to unified format."""
    convs = sample.get('conversations', [])
    turns = []
    for c in convs:
        role = 'user' if c['from'] == 'human' else 'assistant'
        content = c['value'].replace('<image>', '').strip()
        if content:
            turns.append({'role': role, 'content': content})
    return {
        'image_id': sample.get('id', ''),
        'source': 'llava',
        'conversations': turns
    }

def format_vdial_sample(sample):
    """Convert VisualDialog format to unified format (3 turns max)."""
    dialog = sample.get('dialog', [])
    caption = sample.get('caption', '')
    turns = []
    # Start with caption context
    if caption:
        turns.append({'role': 'user', 'content': f'Describe what you see in this image.'})
        turns.append({'role': 'assistant', 'content': caption})
    # Add up to 4 Q-A rounds
    for qa in dialog[:4]:
        q = qa.get('question', '')
        a = qa.get('answer', '')
        if q and a:
            turns.append({'role': 'user', 'content': q})
            turns.append({'role': 'assistant', 'content': a})
    return {
        'image_id': str(sample.get('image_id', '')),
        'source': 'visdial',
        'conversations': turns
    }

# Sample and format
llava_indices = random.sample(range(len(llava_ds)), min(LLAVA_SAMPLES, len(llava_ds)))
vdial_indices = random.sample(range(len(vdial_ds)), min(VDIAL_SAMPLES, len(vdial_ds)))

all_samples = []
for i in llava_indices:
    s = format_llava_sample(llava_ds[i])
    if len(s['conversations']) >= 2:
        all_samples.append(s)

for i in vdial_indices:
    s = format_vdial_sample(vdial_ds[i])
    if len(s['conversations']) >= 2:
        all_samples.append(s)

random.shuffle(all_samples)
split_idx   = int(len(all_samples) * (1 - EVAL_SPLIT))
train_data  = all_samples[:split_idx]
val_data    = all_samples[split_idx:]

print(f'✅ Train samples: {len(train_data)}')
print(f'✅ Val samples:   {len(val_data)}')
print(f'\nSample conversation:')
for turn in train_data[0]['conversations'][:4]:
    print(f"  [{turn['role']}]: {turn['content'][:80]}...")

## Step 5 — Load PaliGemma-3B with QLoRA

In [ ]:
import torch
from transformers import (
    PaliGemmaProcessor,
    PaliGemmaForConditionalGeneration,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'google/paligemma-3b-pt-224'

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # saves extra ~0.4 bits per param
)

print(f'Loading {MODEL_ID}...')
processor = PaliGemmaProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
)
model.config.use_cache = False  # required for gradient checkpointing

# LoRA config — only train attention layers
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,               # rank — higher = more capacity
    lora_alpha=64,      # 2x rank is standard
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5% trainable params (50M / 3B)

# VRAM check
allocated = torch.cuda.memory_allocated() / 1e9
print(f'\nVRAM used: {allocated:.1f} GB')

## Step 6 — Dataset Class & Collator

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import requests
from io import BytesIO

COCO_URL = 'http://images.cocodataset.org/train2017/{:012d}.jpg'
IMAGE_SIZE = 224
MAX_LENGTH = 512

def fetch_coco_image(image_id: str) -> Image.Image:
    """Download COCO image by ID, return blank on failure."""
    try:
        url = COCO_URL.format(int(image_id))
        resp = requests.get(url, timeout=5)
        return Image.open(BytesIO(resp.content)).convert('RGB')
    except Exception:
        return Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color=(128, 128, 128))


class VQADialogDataset(Dataset):
    def __init__(self, samples, processor):
        self.samples   = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        convs  = sample['conversations']

        # Build prompt: interleave user/assistant turns
        prompt_parts = []
        answer = ''
        for i, turn in enumerate(convs):
            if turn['role'] == 'user':
                prompt_parts.append(f"Question: {turn['content']}")
            else:
                if i == len(convs) - 1:   # last turn = target answer
                    answer = turn['content']
                else:
                    prompt_parts.append(f"Answer: {turn['content']}")

        prompt = ' '.join(prompt_parts) + ' Answer:'

        # Fetch image
        image = fetch_coco_image(sample['image_id'])

        # Tokenize
        encoding = self.processor(
            images=image,
            text=prompt,
            return_tensors='pt',
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length',
        )

        # Labels: processor encodes answer separately
        label_encoding = self.processor(
            text=answer,
            return_tensors='pt',
            max_length=64,
            truncation=True,
            padding='max_length',
        )

        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'pixel_values':   encoding['pixel_values'].squeeze(),
            'labels':         label_encoding['input_ids'].squeeze(),
        }


train_dataset = VQADialogDataset(train_data, processor)
val_dataset   = VQADialogDataset(val_data,   processor)

print(f'✅ Train dataset: {len(train_dataset)} samples')
print(f'✅ Val dataset:   {len(val_dataset)} samples')

# Verify one sample
sample = train_dataset[0]
print(f'Input IDs shape:   {sample["input_ids"].shape}')
print(f'Pixel values shape:{sample["pixel_values"].shape}')
print(f'Labels shape:      {sample["labels"].shape}')

## Step 7 — Training

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np

OUTPUT_DIR = '/content/paligemma_vqa_checkpoint'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Batch & gradient
    per_device_train_batch_size=2,       # T4 can handle 2 (vs 1 on RTX 3050)
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,       # effective batch = 8
    gradient_checkpointing=True,

    # Training duration
    num_train_epochs=3,
    warmup_ratio=0.05,

    # Optimizer
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    optim='paged_adamw_8bit',
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Precision
    fp16=True,
    bf16=False,

    # Logging & eval
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    # Misc
    dataloader_num_workers=2,
    report_to='none',           # set to 'wandb' if you want W&B logging
    run_name='paligemma-vqa-multi-turn',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Save loss history for plotting
print('🚀 Starting training...')
train_result = trainer.train()

print(f'\n✅ Training complete!')
print(f'   Total steps:    {train_result.global_step}')
print(f'   Training loss:  {train_result.training_loss:.4f}')
print(f'   Runtime:        {train_result.metrics["train_runtime"] / 60:.1f} min')

## Step 8 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps  = [x['step'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
train_losses = [x['loss'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_steps   = [x['step'] for x in log_history if 'eval_loss' in x]
eval_losses  = [x['eval_loss'] for x in log_history if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_steps, train_losses, label='Train Loss', color='#6366f1', linewidth=2)
ax.plot(eval_steps,  eval_losses,  label='Val Loss',   color='#f59e0b', linewidth=2, marker='o')
ax.set_xlabel('Training Steps')
ax.set_ylabel('Loss')
ax.set_title('PaliGemma-3B QLoRA Training — Multi-Turn VQA')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150)
plt.show()
print(f'✅ Training curves saved to Google Drive')

## Step 9 — Evaluation (Zero-shot vs Fine-tuned)

In [ ]:
from tqdm.auto import tqdm

def generate_answer(model, processor, image, prompt, max_new_tokens=50):
    inputs = processor(
        images=image,
        text=prompt,
        return_tensors='pt',
        max_length=512,
        truncation=True,
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=3,
        )
    decoded = processor.decode(output[0], skip_special_tokens=True)
    # Strip the prompt prefix from output
    if 'Answer:' in decoded:
        decoded = decoded.split('Answer:')[-1].strip()
    return decoded


def exact_match_accuracy(predictions, references):
    correct = sum(
        p.strip().lower() == r.strip().lower()
        for p, r in zip(predictions, references)
    )
    return correct / len(predictions) if predictions else 0.0


# Evaluate on validation set (sample 100 for speed)
EVAL_SAMPLES = min(100, len(val_data))
eval_subset  = random.sample(val_data, EVAL_SAMPLES)

ft_preds, references = [], []

model.eval()
for sample in tqdm(eval_subset, desc='Evaluating fine-tuned'):
    convs = sample['conversations']
    if len(convs) < 2:
        continue
    # Build prompt from all but last assistant turn
    prompt_parts = []
    ground_truth = ''
    for i, turn in enumerate(convs):
        if turn['role'] == 'user':
            prompt_parts.append(f"Question: {turn['content']}")
        elif i < len(convs) - 1:
            prompt_parts.append(f"Answer: {turn['content']}")
        else:
            ground_truth = turn['content']
    prompt = ' '.join(prompt_parts) + ' Answer:'
    image  = fetch_coco_image(sample['image_id'])
    pred   = generate_answer(model, processor, image, prompt)
    ft_preds.append(pred)
    references.append(ground_truth)

ft_accuracy = exact_match_accuracy(ft_preds, references)
print(f'\n✅ Fine-tuned Model Accuracy: {ft_accuracy*100:.1f}%')

## Step 10 — Ablation Study: Conversation History Length

In [ ]:
# Ablation: How much does conversation history help?
# Test the same questions with 0, 1, 2, 3 prior Q-A turns as context

def evaluate_with_history_length(model, processor, samples, max_history_turns):
    preds, refs = [], []
    for sample in samples:
        convs = sample['conversations']
        if len(convs) < 4:  # need at least 2 Q-A pairs
            continue
        # Keep only `max_history_turns` previous pairs + final question
        # Each pair = 2 turns (user + assistant)
        history_turns = convs[:-2]  # all but final Q-A
        final_q = convs[-2]         # final question
        final_a = convs[-1]         # final answer (ground truth)

        # Trim history
        history_turns = history_turns[-(max_history_turns * 2):] if max_history_turns > 0 else []

        # Build prompt
        prompt_parts = []
        for turn in history_turns:
            prefix = 'Question:' if turn['role'] == 'user' else 'Answer:'
            prompt_parts.append(f"{prefix} {turn['content']}")
        prompt_parts.append(f"Question: {final_q['content']} Answer:")
        prompt = ' '.join(prompt_parts)

        image = fetch_coco_image(sample['image_id'])
        pred  = generate_answer(model, processor, image, prompt)
        preds.append(pred)
        refs.append(final_a['content'])

    return exact_match_accuracy(preds, refs)


# Filter samples with enough turns
multi_turn_samples = [s for s in eval_subset if len(s['conversations']) >= 4]
ablation_samples   = multi_turn_samples[:50]

ablation_results = {}
for n_turns in [0, 1, 2, 3]:
    acc = evaluate_with_history_length(model, processor, ablation_samples, n_turns)
    ablation_results[n_turns] = acc
    print(f'History {n_turns} turn(s): {acc*100:.1f}%')

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    [f'{k} turn(s)' for k in ablation_results],
    [v * 100 for v in ablation_results.values()],
    color=['#e2e8f0', '#a5b4fc', '#6366f1', '#4338ca']
)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Ablation: Conversation History Length vs. Accuracy')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/ablation_history.png', dpi=150)
plt.show()
print('✅ Ablation study complete')

## Step 11 — Save Model to Google Drive

In [ ]:
# Save LoRA adapters only (~50-150MB, not the full 3B model)
ADAPTER_SAVE_PATH = f'{SAVE_DIR}/lora_adapters'
model.save_pretrained(ADAPTER_SAVE_PATH)
processor.save_pretrained(ADAPTER_SAVE_PATH)

# Save evaluation results
import json
results = {
    'fine_tuned_accuracy': ft_accuracy,
    'ablation_history':    ablation_results,
    'train_samples':       len(train_data),
    'val_samples':         len(val_data),
    'lora_rank':           32,
    'epochs':              3,
    'model':               MODEL_ID,
}
with open(f'{SAVE_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'✅ LoRA adapters saved to: {ADAPTER_SAVE_PATH}')
print(f'✅ Results saved to: {SAVE_DIR}/results.json')
print(f'\nFiles in Google Drive:')
!ls -lh {SAVE_DIR}

## Step 12 — Interactive Demo
Test your fine-tuned model with your own questions!

In [ ]:
from IPython.display import display

def chat_with_image(image_url_or_path, questions):
    """Multi-turn visual QA demo."""
    if image_url_or_path.startswith('http'):
        image = Image.open(BytesIO(requests.get(image_url_or_path).content)).convert('RGB')
    else:
        image = Image.open(image_url_or_path).convert('RGB')

    display(image.resize((300, 300)))
    print('---')

    history = []
    for question in questions:
        # Build prompt with history
        prompt_parts = []
        for prev_q, prev_a in history:
            prompt_parts.append(f'Question: {prev_q} Answer: {prev_a}')
        prompt_parts.append(f'Question: {question} Answer:')
        prompt = ' '.join(prompt_parts)

        answer = generate_answer(model, processor, image, prompt)
        history.append((question, answer))

        print(f'Q: {question}')
        print(f'A: {answer}')
        print('---')


# Example — replace with any image URL
chat_with_image(
    'http://images.cocodataset.org/train2017/000000000009.jpg',
    [
        'What objects do you see in this image?',
        'What color are they?',
        'How many are there?',
        'Where are they located in the image?',
    ]
)